In [8]:
import numpy as np
from IPython.display import Audio
import scipy.signal
import matplotlib.pyplot as plt

In [9]:
#General variables used through the Assginment
sr = 44100 #Hz
dur = 45 #sekunder
amp = 1

In [10]:

def signal_generator(
        signal_type: str | None, #programming mate thought me to define my variable types, making it easier when implemnting the functions in other document 
        sampling_rate: int | float,
        duration, 
        frequency,
        amplitude,
        phase
    ) -> np.ndarray:
    signal_type = signal_type if not None else 'sinusoid'
    if frequency < 0:
        raise ValueError("frequency below 0")
    

    print("New version loaded 1")
    t = np.linspace(0, duration, int(sampling_rate * duration), endpoint=False)
    #create sinusoid
    sinusoid = amplitude * np.cos(frequency * 2*np.pi * t + phase)

    #create square wave 
    square_wave = []
    for number in sinusoid:
        if number > 0:
            square_wave.append(amplitude)

        elif number < 0:
            square_wave.append(-amplitude)

        elif number == 0:
            square_wave.append(0)

    #create triangel
    triangel_wave = []
    dt = t[1] - t[0] #change in time, is constant, so only one dt needed
    integral_square = 0

    for value in square_wave:
        integral_square += value*dt #value*dt is the integral of the square wave, which is a triangle wave
        triangel_wave.append(integral_square) 
        
    #create saw tooth
    sawtooth_wave = sp.signal.sawtooth(2*np.pi * frequency * t)


    if signal_type == 'sinusoid':
        return sinusoid
    elif signal_type == 'square wave':
        return square_wave
    elif signal_type == 'triangel wave':
        return np.array(triangel_wave)
    elif signal_type == 'saw tooth':
        return np.array(sawtooth_wave)
    else:
        raise ValueError("Wrong type hilsen sigs")

def noise_generator(samplingrate, duration, amplitude):
    n = samplingrate * duration
    noise = (np.random.rand(n) - np.random.rand(n)) * amplitude
    return noise

In [11]:
#filters

def plotFilter(b, a, title=""):
    f, h = scipy.signal.freqz(b, a, worN=sr, fs = sr)
    amp = np.abs(h)
    phase = np.angle(h)

    fig, axs = plt.subplots(2, figsize=(8,6), sharex=True)

    fig.suptitle(title)

    axs[0].plot(f, 20*np.log10(amp))
    axs[0].set_xscale("log")
    axs[0].set_title("Amplitude response")
    axs[0].set_ylabel("Gain [dB]")
    axs[0].set_ylim(-40, 5)
    axs[0].grid()
    axs[0].legend()


    axs[1].plot(f, np.degrees(phase))
    axs[1].set_title("Phase response")
    axs[1].set_ylabel("Phase [deg]")
    axs[1].set_xlabel("Frequency [Hz]")
    axs[1].set_xlim(0.1, sr/2)
    axs[1].grid()
    fig.tight_layout()
    plt.show()

#lowpass
def lowpass(signal, order, fqcut, sr, plot = False):
    b, a = scipy.signal.butter(order, fqcut, btype='low', fs=sr)

    #Fq respons
    if plot:
        plotFilter(b, a, title = "Lowpass Filter")

    return scipy.signal.lfilter(b,a, signal)


#Highpass
def highpass(signal, order, fqcut, sr, plot = False):
    b, a = scipy.signal.butter(order, fqcut, btype='high', fs=sr)

    if plot:
           plotFilter(b, a, title = "Highpass Filter")
    return scipy.signal.lfilter(b,a, signal)

#Bandpass
def bandpass(signal, order, lowcut, highcut, sr, plot = False):
    b, a = scipy.signal.butter(order, [lowcut,highcut], btype='band', fs=sr)

    if plot:
           plotFilter(b, a, title ="Bandpass Filter")

    return scipy.signal.lfilter(b, a, signal)


#Allpass (Claude was used to imporve this function)
def allpass(signal, sr, c, cascade, plot = False): 

    b = [c, 1]
    a = [1, c]

    for i in range(cascade):
        signal = scipy.signal.lfilter(b, a, signal)
        
    if plot: #Making phase shift plot as allpass dosent change frequency but only phase
        f, h = scipy.signal.freqz(b, a, worN=8000)

        radian = np.unwrap(np.angle(h))
        degrees = np.degrees(radian)

        normalised = f / np.pi
        plt.plot(normalised, degrees, label="Allpass Fase")

        plt.title("Phase shift")
        plt.xlabel("Normalized Frequency [* pi rad/sample]")
        plt.ylabel("Phase shift [degrees]")
        plt.grid()
        plt.legend()
        plt.show()

    return signal

In [ ]:
#Spaceship noise

noise = noise_generator(sr, dur, 1)

lowNoise = lowpass(noise, 4, 180, sr, False)

Audio(lowNoise, rate=sr)

array([ 0.42084841,  0.56908882,  0.1704877 , ..., -0.46646705,
       -0.6223821 , -0.22141807], shape=(1984500,))